<a href="https://colab.research.google.com/github/ladams1204/SportsBookEdge/blob/main/06_odds_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!pip install nba_api xgboost shap --quiet

import sqlite3
import pandas as pd
import numpy as np
import requests
import json
import time
from datetime import datetime, timedelta

from google.colab import userdata

# Get the API key
odds_api_key = userdata.get('ODDS_API_KEY').strip()
print("Odds API key loaded:", odds_api_key[:8] + "..." + odds_api_key[-4:])

Odds API key loaded: 1233c426...bf5a


In [29]:
# Check what's available for NBA today
url = "https://api.the-odds-api.com/v4/sports/basketball_nba/events"
response = requests.get(url, params={
    "apiKey": odds_api_key
})

print("Status:", response.status_code)
print("Credits remaining this month:", response.headers.get("x-requests-remaining"))

events = response.json()
print(f"\nGames available: {len(events)}")

if len(events) > 0:
    print("\nUpcoming games (first 5):")
    for e in events[:5]:
        print(f"  {e['id'][:12]}... {e['away_team']} @ {e['home_team']} — {e['commence_time']}")
else:
    print("\nNo games found — it might be the offseason, or no games scheduled today.")

Status: 200
Credits remaining this month: 490

Games available: 8

Upcoming games (first 5):
  bee2c9ea4bf8... Atlanta Hawks @ New York Knicks — 2026-04-18T22:18:39Z
  1a9383105929... Houston Rockets @ Los Angeles Lakers — 2026-04-19T00:40:00Z
  00d1efd43d07... Philadelphia 76ers @ Boston Celtics — 2026-04-19T17:10:00Z
  ef6021edff80... Phoenix Suns @ Oklahoma City Thunder — 2026-04-19T19:40:00Z
  e777e2f049a5... Orlando Magic @ Detroit Pistons — 2026-04-19T22:40:00Z


In [30]:
# Your chosen games
target_game_ids = {
    '1a9383105929': 'Rockets @ Lakers',
    '00d1efd43d07': 'Sixers @ Celtics',
    'ef6021edff80': 'Suns @ Thunder',
    'e777e2f049a5': 'Magic @ Pistons'
}

# Note: the 12-char truncated IDs we see in output are the start of longer IDs
# We need the FULL IDs from the events response

# Grab full game IDs
target_games = [e for e in events if e['id'][:12] in target_game_ids]

print(f"Matched {len(target_games)} games:")
for g in target_games:
    print(f"  {g['id']}")
    print(f"    {g['away_team']} @ {g['home_team']} — {g['commence_time']}")

Matched 4 games:
  1a9383105929a09e5662227ad45554b8
    Houston Rockets @ Los Angeles Lakers — 2026-04-19T00:40:00Z
  00d1efd43d0790297fb68684a9171191
    Philadelphia 76ers @ Boston Celtics — 2026-04-19T17:10:00Z
  ef6021edff80a85fe2dae5cf3530d9d4
    Phoenix Suns @ Oklahoma City Thunder — 2026-04-19T19:40:00Z
  e777e2f049a5c85aed96a220a2cfe924
    Orlando Magic @ Detroit Pistons — 2026-04-19T22:40:00Z


In [31]:
def pull_player_points_props(game_id, game_label):
    """Pull player points over/under props for a specific NBA game."""
    url = f"https://api.the-odds-api.com/v4/sports/basketball_nba/events/{game_id}/odds"

    params = {
        "apiKey": odds_api_key,
        "regions": "us",              # US sportsbooks only
        "markets": "player_points",   # the prop market we care about
        "oddsFormat": "american",     # e.g. -110, +120 format
        "bookmakers": "draftkings,fanduel,betmgm"  # limit to major books to save credits
    }

    response = requests.get(url, params=params)
    credits_left = response.headers.get("x-requests-remaining")

    if response.status_code != 200:
        print(f"  ✗ {game_label}: HTTP {response.status_code}")
        print(f"    Response: {response.text[:200]}")
        return None, credits_left

    data = response.json()
    print(f"  ✓ {game_label}: {len(data.get('bookmakers', []))} books responded — credits: {credits_left}")
    return data, credits_left


# Pull all 4 games
all_props = {}
for g in target_games:
    label = target_game_ids[g['id'][:12]]
    data, credits = pull_player_points_props(g['id'], label)
    if data:
        all_props[g['id']] = {
            'label': label,
            'home_team': g['home_team'],
            'away_team': g['away_team'],
            'commence_time': g['commence_time'],
            'data': data
        }
    time.sleep(0.5)  # be polite

print(f"\nPulled props for {len(all_props)}/{len(target_games)} games")
print(f"Credits remaining: {credits}")

  ✓ Rockets @ Lakers: 2 books responded — credits: 489
  ✓ Sixers @ Celtics: 3 books responded — credits: 488
  ✓ Suns @ Thunder: 3 books responded — credits: 487
  ✓ Magic @ Pistons: 3 books responded — credits: 486

Pulled props for 4/4 games
Credits remaining: 486


In [32]:
import json

# Save everything we pulled to a JSON file
with open('prop_data_cache.json', 'w') as f:
    # Convert the data into a serializable format
    cache = {gid: info for gid, info in all_props.items()}
    json.dump(cache, f, indent=2, default=str)

print("Cached raw prop data to prop_data_cache.json")
print(f"File size: approximately {len(json.dumps(cache))} characters")

# Verify we can load it back
with open('prop_data_cache.json', 'r') as f:
    loaded = json.load(f)

print(f"Verification: loaded {len(loaded)} games back from cache")

Cached raw prop data to prop_data_cache.json
File size: approximately 26378 characters
Verification: loaded 4 games back from cache


In [33]:
# Parse the raw prop data into a flat DataFrame
rows = []

for game_id, info in all_props.items():
    game_label = info['label']
    data = info['data']

    for book in data.get('bookmakers', []):
        book_name = book['key']

        for market in book.get('markets', []):
            if market['key'] != 'player_points':
                continue

            player_lines = {}

            for outcome in market['outcomes']:
                player_name = outcome['description']
                side = outcome['name']
                line = outcome['point']
                odds = outcome['price']

                if player_name not in player_lines:
                    player_lines[player_name] = {'line': line}

                if side == 'Over':
                    player_lines[player_name]['over_odds'] = odds
                else:
                    player_lines[player_name]['under_odds'] = odds

            for player_name, details in player_lines.items():
                rows.append({
                    'game': game_label,
                    'book': book_name,
                    'player_name': player_name,
                    'line': details['line'],
                    'over_odds': details.get('over_odds'),
                    'under_odds': details.get('under_odds')
                })

props_df = pd.DataFrame(rows)
print(f"Total prop rows: {len(props_df)}")
print(f"Unique players with props: {props_df['player_name'].nunique()}")
print(f"Books: {props_df['book'].unique()}")
print(f"\nSample:")
print(props_df.head(15).to_string(index=False))

Total prop rows: 145
Unique players with props: 57
Books: ['draftkings' 'fanduel' 'betmgm']

Sample:
            game       book       player_name  line  over_odds  under_odds
Rockets @ Lakers draftkings      LeBron James  25.5       -115        -110
Rockets @ Lakers draftkings    Alperen Sengun  21.5       -121        -105
Rockets @ Lakers draftkings     Amen Thompson  19.5       -125        -102
Rockets @ Lakers draftkings   Jabari Smith Jr  16.5       -110        -116
Rockets @ Lakers draftkings     Reed Sheppard  15.5       -103        -123
Rockets @ Lakers draftkings     Rui Hachimura  14.5       -105        -122
Rockets @ Lakers draftkings      Luke Kennard  12.5       -115        -110
Rockets @ Lakers draftkings     Deandre Ayton  12.5       -106        -120
Rockets @ Lakers draftkings      Marcus Smart  11.5       -109        -117
Rockets @ Lakers draftkings        Tari Eason  11.5       -106        -121
Rockets @ Lakers draftkings      Jake LaRavia   7.5       -111        -115

In [34]:
consensus = (
    props_df.groupby(['game', 'player_name'])
            .agg(
                consensus_line=('line', 'median'),
                book_count=('book', 'nunique'),
                avg_over_odds=('over_odds', 'mean'),
                avg_under_odds=('under_odds', 'mean')
            )
            .reset_index()
            .sort_values(['game', 'consensus_line'], ascending=[True, False])
)

print(f"Consensus lines for {len(consensus)} player-game combos:")
print(consensus.to_string(index=False))

Consensus lines for 57 player-game combos:
            game             player_name  consensus_line  book_count  avg_over_odds  avg_under_odds
 Magic @ Pistons         Cade Cunningham            26.5           3    -115.000000     -113.000000
 Magic @ Pistons          Paolo Banchero            22.5           3    -117.333333     -111.666667
 Magic @ Pistons             Jalen Duren            20.5           3    -115.333333     -113.333333
 Magic @ Pistons            Desmond Bane            19.5           3    -114.333333     -114.000000
 Magic @ Pistons            Franz Wagner            17.5           3    -107.333333     -122.000000
 Magic @ Pistons             Jalen Suggs            13.5           3    -117.333333     -111.666667
 Magic @ Pistons           Tobias Harris            13.5           3    -106.666667     -122.333333
 Magic @ Pistons         Duncan Robinson            11.5           3     -38.666667     -124.333333
 Magic @ Pistons           Anthony Black             9.5 

In [35]:
# Pull recent data for our 5 matched players + compute simple predictions
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.static import players

# Players in tonight's slate who we have data for
tonight_players = [
    ('LeBron James', 'Rockets @ Lakers'),
    ('Jayson Tatum', 'Sixers @ Celtics'),
    ('Shai Gilgeous-Alexander', 'Suns @ Thunder'),
    ('Devin Booker', 'Suns @ Thunder'),
    ('Paul George', 'Sixers @ Celtics')
]

predictions = []

for player_name, game in tonight_players:
    matches = players.find_players_by_full_name(player_name)
    if not matches:
        print(f"✗ {player_name}: not found")
        continue

    player_id = matches[0]['id']

    # Pull this season's game log
    log = playergamelog.PlayerGameLog(player_id=player_id, season='2025-26')
    df = log.get_data_frames()[0]

    if len(df) < 5:
        print(f"✗ {player_name}: only {len(df)} games this season")
        continue

    # Recent points (most recent games)
    last_5 = df['PTS'].head(5).mean()
    last_10 = df['PTS'].head(10).mean()
    last_20 = df['PTS'].head(20).mean() if len(df) >= 20 else df['PTS'].mean()
    season_avg = df['PTS'].mean()

    predictions.append({
        'player_name': player_name,
        'game': game,
        'last_5_avg': round(last_5, 1),
        'last_10_avg': round(last_10, 1),
        'last_20_avg': round(last_20, 1),
        'season_avg': round(season_avg, 1),
        'our_prediction': round(last_20, 1)  # use 20-game as our "prediction"
    })

    print(f"✓ {player_name}: L5={last_5:.1f}, L10={last_10:.1f}, L20={last_20:.1f}, Season={season_avg:.1f}")
    time.sleep(0.6)

pred_df = pd.DataFrame(predictions)
print("\n--- Our Predictions ---")
print(pred_df.to_string(index=False))

✓ LeBron James: L5=23.0, L10=19.9, L20=19.8, Season=20.9
✓ Jayson Tatum: L5=23.6, L10=22.4, L20=21.8, Season=21.8
✓ Shai Gilgeous-Alexander: L5=28.0, L10=29.0, L20=28.9, Season=31.1
✓ Devin Booker: L5=30.8, L10=27.7, L20=29.0, Season=26.1
✓ Paul George: L5=15.0, L10=21.0, L20=18.6, Season=17.3

--- Our Predictions ---
            player_name             game  last_5_avg  last_10_avg  last_20_avg  season_avg  our_prediction
           LeBron James Rockets @ Lakers        23.0         19.9         19.8        20.9            19.8
           Jayson Tatum Sixers @ Celtics        23.6         22.4         21.8        21.8            21.8
Shai Gilgeous-Alexander   Suns @ Thunder        28.0         29.0         29.0        31.1            29.0
           Devin Booker   Suns @ Thunder        30.8         27.7         29.0        26.1            29.0
            Paul George Sixers @ Celtics        15.0         21.0         18.6        17.3            18.6


In [36]:
# Merge predictions with market lines
comparison = pred_df.merge(
    consensus[['player_name', 'consensus_line', 'book_count', 'avg_over_odds', 'avg_under_odds']],
    on='player_name',
    how='inner'
)

# Compute the "edge": how much does our prediction differ from Vegas?
comparison['diff_vs_line'] = comparison['our_prediction'] - comparison['consensus_line']
comparison['lean'] = comparison['diff_vs_line'].apply(
    lambda x: 'OVER' if x > 0 else ('UNDER' if x < 0 else 'PUSH')
)

# Sort by absolute difference (biggest disagreements first)
comparison['abs_diff'] = comparison['diff_vs_line'].abs()
comparison = comparison.sort_values('abs_diff', ascending=False)

print("=== MODEL vs VEGAS ===\n")
print(comparison[[
    'player_name', 'game', 'our_prediction', 'consensus_line',
    'diff_vs_line', 'lean', 'avg_over_odds', 'avg_under_odds', 'book_count'
]].to_string(index=False))

print("\n--- Interpretation ---")
print("diff_vs_line > 0 → our model thinks OVER is more likely")
print("diff_vs_line < 0 → our model thinks UNDER is more likely")
print("A 2+ point difference with clean odds is notable, but small sample warning applies.")

=== MODEL vs VEGAS ===

            player_name             game  our_prediction  consensus_line  diff_vs_line  lean  avg_over_odds  avg_under_odds  book_count
           LeBron James Rockets @ Lakers            19.8            25.5          -5.7 UNDER    -108.500000          -117.5           2
           Devin Booker   Suns @ Thunder            29.0            23.5           5.5  OVER    -120.666667          -108.0           3
           Jayson Tatum Sixers @ Celtics            21.8            23.5          -1.7 UNDER    -113.333333          -116.0           3
Shai Gilgeous-Alexander   Suns @ Thunder            29.0            30.5          -1.5 UNDER    -108.666667          -120.0           3
            Paul George Sixers @ Celtics            18.6            19.5          -0.9 UNDER     -41.333333          -121.0           3

--- Interpretation ---
diff_vs_line > 0 → our model thinks OVER is more likely
diff_vs_line < 0 → our model thinks UNDER is more likely
A 2+ point difference w

In [37]:
# Save today's predictions to a tracking log
tracking_file = 'paper_trades.csv'

comparison['date'] = datetime.now().strftime('%Y-%m-%d')
comparison['actual_points'] = None  # To fill in after games complete
comparison['result'] = None

# Try to append to existing file, else create new
try:
    existing = pd.read_csv(tracking_file)
    combined = pd.concat([existing, comparison], ignore_index=True)
except FileNotFoundError:
    combined = comparison.copy()

combined.to_csv(tracking_file, index=False)
print(f"Saved {len(comparison)} predictions to {tracking_file}")
print(f"Total tracked predictions: {len(combined)}")

Saved 5 predictions to paper_trades.csv
Total tracked predictions: 31


In [38]:
# Expanded list — more players across all 4 games
# I've focused on players with 3-book consensus and symmetric odds
# (skipping weird lines like Paul George -41 which indicate news-driven moves)

expanded_players = [
    # Rockets @ Lakers
    ('LeBron James', 'Rockets @ Lakers'),
    ('Alperen Sengun', 'Rockets @ Lakers'),
    ('Amen Thompson', 'Rockets @ Lakers'),
    ('Jabari Smith Jr', 'Rockets @ Lakers'),
    ('Rui Hachimura', 'Rockets @ Lakers'),
    ('Luke Kennard', 'Rockets @ Lakers'),
    ('Deandre Ayton', 'Rockets @ Lakers'),

    # Sixers @ Celtics
    ('Tyrese Maxey', 'Sixers @ Celtics'),
    ('Jaylen Brown', 'Sixers @ Celtics'),
    ('Jayson Tatum', 'Sixers @ Celtics'),
    ('Derrick White', 'Sixers @ Celtics'),
    ('Kelly Oubre Jr', 'Sixers @ Celtics'),
    ('Payton Pritchard', 'Sixers @ Celtics'),
    ('Nikola Vucevic', 'Sixers @ Celtics'),

    # Suns @ Thunder
    ('Shai Gilgeous-Alexander', 'Suns @ Thunder'),
    ('Devin Booker', 'Suns @ Thunder'),
    ('Jalen Green', 'Suns @ Thunder'),
    ('Jalen Williams', 'Suns @ Thunder'),
    ('Chet Holmgren', 'Suns @ Thunder'),
    ('Dillon Brooks', 'Suns @ Thunder'),
    ('Grayson Allen', 'Suns @ Thunder'),
    ('Luguentz Dort', 'Suns @ Thunder'),

    # Magic @ Pistons
    ('Cade Cunningham', 'Magic @ Pistons'),
    ('Paolo Banchero', 'Magic @ Pistons'),
    ('Jalen Duren', 'Magic @ Pistons'),
    ('Desmond Bane', 'Magic @ Pistons'),
    ('Franz Wagner', 'Magic @ Pistons'),
    ('Jalen Suggs', 'Magic @ Pistons'),
    ('Tobias Harris', 'Magic @ Pistons'),
    ('Wendell Carter Jr', 'Magic @ Pistons'),
    ('Ausar Thompson', 'Magic @ Pistons'),
]

print(f"Will generate predictions for {len(expanded_players)} players")

Will generate predictions for 31 players


In [39]:
predictions = []
failed = []

for player_name, game in expanded_players:
    try:
        matches = players.find_players_by_full_name(player_name)
        if not matches:
            failed.append(f"{player_name}: not found")
            continue

        player_id = matches[0]['id']

        log = playergamelog.PlayerGameLog(player_id=player_id, season='2025-26')
        df = log.get_data_frames()[0]

        if len(df) < 5:
            failed.append(f"{player_name}: only {len(df)} games")
            continue

        # Compute averages (df is sorted most-recent first)
        last_5 = df['PTS'].head(5).mean()
        last_10 = df['PTS'].head(10).mean()
        last_20 = df['PTS'].head(20).mean() if len(df) >= 20 else df['PTS'].mean()
        season_avg = df['PTS'].mean()
        games_played = len(df)

        predictions.append({
            'player_name': player_name,
            'game': game,
            'games_played': games_played,
            'last_5_avg': round(last_5, 1),
            'last_10_avg': round(last_10, 1),
            'last_20_avg': round(last_20, 1),
            'season_avg': round(season_avg, 1),
            'our_prediction': round(last_20, 1)
        })

        print(f"✓ {player_name}: L5={last_5:.1f}, L20={last_20:.1f}, Season={season_avg:.1f}")
        time.sleep(0.6)

    except Exception as e:
        failed.append(f"{player_name}: {str(e)[:60]}")

print(f"\nSuccessful: {len(predictions)}/{len(expanded_players)}")
if failed:
    print("Failed:")
    for f in failed:
        print(f"  {f}")

pred_df_expanded = pd.DataFrame(predictions)

✓ LeBron James: L5=23.0, L20=19.8, Season=20.9
✓ Alperen Sengun: L5=17.0, L20=20.7, Season=20.4
✓ Amen Thompson: L5=24.2, L20=20.8, Season=18.3
✓ Jabari Smith Jr: L5=19.2, L20=16.6, Season=15.8
✓ Rui Hachimura: L5=16.6, L20=11.0, Season=11.5
✓ Luke Kennard: L5=12.2, L20=8.4, Season=8.4
✓ Deandre Ayton: L5=13.8, L20=11.6, Season=12.5
✓ Tyrese Maxey: L5=23.0, L20=26.9, Season=28.3
✓ Jaylen Brown: L5=30.6, L20=27.4, Season=28.7
✓ Jayson Tatum: L5=23.6, L20=21.8, Season=21.8
✓ Derrick White: L5=11.2, L20=14.8, Season=16.5
✓ Kelly Oubre Jr: L5=10.0, L20=14.1, Season=14.1
✓ Payton Pritchard: L5=17.8, L20=17.1, Season=17.0
✓ Nikola Vucevic: L5=6.0, L20=11.2, Season=15.1
✓ Shai Gilgeous-Alexander: L5=28.0, L20=28.9, Season=31.1
✓ Devin Booker: L5=30.8, L20=29.0, Season=26.1
✓ Jalen Green: L5=14.6, L20=20.6, Season=17.8
✓ Jalen Williams: L5=16.6, L20=16.9, Season=17.1
✓ Chet Holmgren: L5=17.2, L20=16.3, Season=17.1
✓ Dillon Brooks: L5=15.6, L20=19.3, Season=20.2
✓ Grayson Allen: L5=9.0, L20=16.

In [40]:
# Merge with the consensus lines we already have
comparison_expanded = pred_df_expanded.merge(
    consensus[['player_name', 'consensus_line', 'book_count', 'avg_over_odds', 'avg_under_odds']],
    on='player_name',
    how='inner'
)

comparison_expanded['diff_vs_line'] = comparison_expanded['our_prediction'] - comparison_expanded['consensus_line']
comparison_expanded['lean'] = comparison_expanded['diff_vs_line'].apply(
    lambda x: 'OVER' if x > 0 else ('UNDER' if x < 0 else 'PUSH')
)
comparison_expanded['abs_diff'] = comparison_expanded['diff_vs_line'].abs()

# Filter to only well-supported lines (3 books, roughly symmetric odds)
# This screens out noise/injury-driven lines
comparison_clean = comparison_expanded[
    (comparison_expanded['book_count'] >= 2) &
    (comparison_expanded['avg_over_odds'] >= -130) &
    (comparison_expanded['avg_over_odds'] <= -100) &
    (comparison_expanded['avg_under_odds'] >= -130) &
    (comparison_expanded['avg_under_odds'] <= -100)
].copy()

comparison_clean = comparison_clean.sort_values('abs_diff', ascending=False)

print("=== EXPANDED MODEL vs VEGAS (clean markets only) ===\n")
print(comparison_clean[[
    'player_name', 'game', 'our_prediction', 'consensus_line',
    'diff_vs_line', 'lean', 'avg_over_odds', 'avg_under_odds'
]].to_string(index=False))

print(f"\n{len(comparison_clean)} clean comparisons (filtered out injury/stale markets)")
print(f"Biggest disagreements at the top — those are your 'edges' candidates")

=== EXPANDED MODEL vs VEGAS (clean markets only) ===

            player_name             game  our_prediction  consensus_line  diff_vs_line  lean  avg_over_odds  avg_under_odds
          Grayson Allen   Suns @ Thunder            16.3             9.5           6.8  OVER    -110.500000     -118.000000
        Cade Cunningham  Magic @ Pistons            20.6            26.5          -5.9 UNDER    -115.000000     -113.000000
           LeBron James Rockets @ Lakers            19.8            25.5          -5.7 UNDER    -108.500000     -117.500000
           Devin Booker   Suns @ Thunder            29.0            23.5           5.5  OVER    -120.666667     -108.000000
           Jaylen Brown Sixers @ Celtics            27.4            24.5           2.9  OVER    -116.333333     -113.000000
         Nikola Vucevic Sixers @ Celtics            11.2             8.5           2.7  OVER    -110.666667     -118.666667
            Jalen Green   Suns @ Thunder            20.6            18.5      

In [41]:
# Save the expanded set to paper_trades.csv
comparison_clean['date'] = datetime.now().strftime('%Y-%m-%d')
comparison_clean['actual_points'] = None
comparison_clean['result'] = None

# Append to the tracking file
try:
    existing = pd.read_csv('paper_trades.csv')
    combined = pd.concat([existing, comparison_clean], ignore_index=True)
    # Deduplicate in case you re-ran
    combined = combined.drop_duplicates(
        subset=['date', 'player_name', 'game'], keep='last'
    )
except FileNotFoundError:
    combined = comparison_clean.copy()

combined.to_csv('paper_trades.csv', index=False)
print(f"Total picks now in tracking log: {len(combined)}")
print(f"Today's picks: {len(comparison_clean)}")

Total picks now in tracking log: 26
Today's picks: 24
